# Import Libraries and Load Data

Import pandas, NumPy, scikit-learn, and any visualization libraries. Load the raw employee attrition dataset from data/raw/data.csv.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Load the raw employee attrition dataset
df = pd.read_csv('data/raw/data.csv')
df.head()

# Inspect and Validate Data

Display dataset shape, column types, missing values, and target distribution. Apply basic validation rules to identify issues in the raw data.

In [ ]:
print('Dataset shape:', df.shape)
print('Column types:')
print(df.dtypes)
print('Missing values:')
print(df.isnull().sum())
print('Target distribution:')
print(df['Attrition'].value_counts())  # assuming the column is 'Attrition'

# Preprocess and Clean Features

Handle missing values, encode categorical variables, and normalize numeric features in preparation for modeling.

In [ ]:
# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

# Encode categorical variables
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

# Normalize numeric features
scaler = StandardScaler()
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Feature Engineering

Create derived features and select relevant predictors for attrition prediction, using techniques like one-hot encoding and feature scaling.

In [ ]:
# Feature selection - drop irrelevant columns if any
# Assuming no irrelevant columns, or add logic to select features
# For example, correlation analysis
correlation = df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.show()

# Select features with high correlation to target
target_corr = correlation['Attrition'].abs().sort_values(ascending=False)
selected_features = target_corr[target_corr > 0.1].index.tolist()
selected_features.remove('Attrition')  # remove target
print('Selected features:', selected_features)

# Train-Test Split

Split the dataset into training and testing sets while preserving class balance, and prepare data for model training.

In [ ]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train a Classification Model

Train a machine learning model such as logistic regression or random forest on the training data to predict employee attrition.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate Model Performance

Evaluate the trained model using accuracy, precision, recall, F1 score, and confusion matrix on the test set.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1 Score:', f1_score(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

# Run Inference on New Samples

Demonstrate predicting attrition for new employee records and interpreting model outputs.

In [ ]:
# Example new sample
new_sample = X_test.iloc[0:1]
prediction = model.predict(new_sample)
print('Prediction:', prediction)
print('Probability:', model.predict_proba(new_sample))

# Save Artifacts and Model

Save the trained model and any preprocessing artifacts to disk, matching the repository structure for production use.

In [ ]:
import joblib

# Save model
joblib.dump(model, 'models/model.pkl')

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')